In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
nu = 2.0  # Degrees of freedom for Student's t
d2 = np.linspace(0, 20, 400)  # Squared Mahalanobis distance

# Weights
# Gaussian/Softmax weight: exp(-d^2 / nu)
w_softmax = np.exp(-d2 / nu)
Student's t / AFA rational weight: (1 + d^2 / nu)^-1
w_afa = 1.0 / (1.0 + d2 / nu)
# d = 256
# kappa = (nu + d) / d
# w_afa = (1.0 + d2 / nu)**(-kappa)

# Create side-by-side subplots for Linear and Log scales
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Linear scale (The Heavy-Tail Effect)
ax1.plot(d2, w_softmax, label='Softmax (Exponential)', color='#E74C3C', linewidth=2.5)
ax1.plot(d2, w_afa, label="Student's t", color='#2980B9', linewidth=2.5)
ax1.set_xlabel(r'Squared Mahalanobis Distance ($d^2$)', fontsize=12)
ax1.set_ylabel('Influence Weight $w(d^2)$', fontsize=12)
ax1.set_title('Linear Scale: Evidence Integration', fontsize=14)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# Plot 2: Log scale (The Softmax Cliff)
# We plot the log-weights to illustrate the dynamic range and gradient stability
ax2.plot(d2, np.log(w_softmax + 1e-12), label='Softmax (Exponential)', color='#E74C3C', linewidth=2.5)
ax2.plot(d2, np.log(w_afa + 1e-12), label="Student's t", color='#2980B9', linewidth=2.5)
ax2.set_xlabel(r'Squared Mahalanobis Distance ($d^2$)', fontsize=12)
ax2.set_ylabel(r'Log Influence Weight $\log w(d^2)$', fontsize=12)
ax2.set_title('Log Scale: Numerical Stability', fontsize=14)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('robustness_comparison.png', dpi=300)

Influence Function Comparison. Comparison between the Gaussian (Softmax) and Student’s $t$ (AFA) influence functions $w(d^2)$ across linear and logarithmic scales.
* Left (Linear): Illustrates the Heavy-Tail Effect. Standard Softmax (red) aggressively suppresses residuals, leading to a "winner-take-all" collapse. The AFA observer (blue) maintains a non-zero influence for "surprising" but informative tokens, enabling multi-modal evidence integration.
* Right (Log): Highlights the Softmax Cliff. The exponential decay of standard attention leads to a rapid loss of dynamic range in high dimensions ($d \gg 1$). AFA’s logarithmic decay preserves gradient stability and prevents rank collapse, ensuring the observer remains sensitive to distal innovations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
nu = 2.0  # Degrees of freedom
d2 = np.linspace(0, 5, 400)

# Logits (Negative Log-Likelihoods)
# Gaussian: -d^2 / nu
# Student's t: -log(1 + d^2 / nu)
logit_gaussian = -d2 / nu
logit_t = -np.log(1 + d2 / nu)

# Weights
weight_gaussian = np.exp(logit_gaussian)
weight_t = np.exp(logit_t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: The "Flatness" of the Logit (Log-Likelihood)
ax1.plot(d2, logit_gaussian, label='Gaussian Logit ($-d^2/\\nu$)', color='#E74C3C', linestyle='--')
ax1.plot(d2, logit_t, label='Student\'s $t$ Logit ($-\\log(1 + d^2/\\nu)$)', color='#2980B9', linewidth=2)
ax1.set_xlabel('Squared Mahalanobis Distance $d^2$', fontsize=12)
ax1.set_ylabel('Logit Value', fontsize=12)
ax1.set_title('Logit Curvature: Avoiding Over-Sensitivity', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Weight comparison near zero
ax2.plot(d2, weight_gaussian, label='Gaussian Weight', color='#E74C3C', linestyle='--')
ax2.plot(d2, weight_t, label='Student\'s $t$ Weight', color='#2980B9', linewidth=2)
ax2.set_xlabel('Squared Mahalanobis Distance $d^2$', fontsize=12)
ax2.set_ylabel('Influence Weight $w(d^2)$', fontsize=12)
ax2.set_title('Weight Compression: The Noise Floor Effect', fontsize=14)
ax2.set_ylim(0.2, 1.05)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('flatness_near_zero.png', dpi=300)
print("Plot saved as flatness_near_zero.png")

Figure: Local Stability of the AFA Observer. This figure illustrates the "non-committal" inductive bias of the Student’s $t$ observer when handling measurements near the intrinsic noise floor.Left (Logit Curvature): Contrasts the linear gain of the Gaussian logit with the logarithmic compression of the Student’s $t$ logit. The flattening of the curve near $d^2 = 0$ represents a reduced sensitivity to infinitesimal differences in match quality. This ensures that the attention mechanism does not over-react to stochastic fluctuations when multiple tokens are statistically consistent with the predicted state.Right (Weight Compression): Demonstrates the resulting influence weights. While standard Softmax (red) aggressively differentiates between "good" matches, AFA (blue) exhibits a broader, compressed peak. This "noise floor dampening" ensures that high-confidence evidence is integrated more uniformly, preventing the model from prematurely collapsing attention onto a single token before the signal-to-noise ratio justifies high selectivity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d2 = np.linspace(0, 5, 500)
nu = 2.0

# Logit Curvature (Absolute value of the second derivative of the penalty)
# Gaussian Penalty: d^2 / nu  -> Curvature is 0
# Student's t Penalty: log(1 + d^2 / nu) -> Curvature is 1 / (nu + d2)^2
logit_curv_gaussian = np.zeros_like(d2)
logit_curv_t = 1.0 / (nu + d2)**2

plt.figure(figsize=(8, 5))
plt.plot(d2, logit_curv_gaussian, label='Gaussian (Linear Penalty Rate)', color='#E74C3C', linestyle='--')
plt.plot(d2, logit_curv_t, label="Student's t (Diminishing Penalty)", color='#2980B9', linewidth=2.5)

plt.title('Penalty Sensitivity: The "Slowing Down" Effect', fontsize=14)
plt.xlabel('Squared Mahalanobis Distance $d^2$')
plt.ylabel('Penalty Curvature $|d^2L/d(d^2)^2|$')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Comparison: Two tokens with a fixed distance gap (e.g., token 1 is slightly better than token 2)
# We vary the absolute scale of the distances to see the "Sensitivity"
delta = 0.5 
d2_values = np.linspace(0.1, 10, 100)
nu = 2.0

# Calculate Weight Ratios: W(d^2) / W(d^2 + delta)
# Gaussian: exp(-d2) / exp(-(d2+delta)) = exp(delta) -> CONSTANT SENSITIVITY
ratio_gaussian = np.exp(delta / nu) * np.ones_like(d2_values)

# Student's t: (1 + (d2+delta)/nu) / (1 + d2/nu) -> DECAYING SENSITIVITY
ratio_t = (1 + (d2_values + delta) / nu) / (1 + d2_values / nu)

plt.figure(figsize=(8, 5))
plt.plot(d2_values, ratio_gaussian, label='Gaussian (Standard Attention)', color='#E74C3C', linestyle='--', linewidth=2)
plt.plot(d2_values, ratio_t, label="AFA (Student's t)", color='#2980B9', linewidth=3)

plt.axhline(1.0, color='black', alpha=0.2)
plt.xlabel(r'Absolute Match Quality ($d^2$)', fontsize=12)
plt.ylabel('Sensitivity (Weight Ratio $w_1 / w_2$)', fontsize=12)
plt.title('Winner-Take-All vs. Non-Committal Attention', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Annotation for the "Noise Floor"
plt.annotate('Noise Floor Dampening', xy=(0.5, 1.15), xytext=(3, 1.25),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))

plt.savefig('noise_floor_dampening.png', dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Range focused on the origin (near-perfect matches)
d2 = np.linspace(0, 2, 400)
nu = 2.0

# Logits (Attention Energies)
# Gaussian: L = -d^2 / nu (Linear gain)
# Student's t: L = -log(1 + d^2 / nu) (Compressed gain)
logit_gaussian = -d2 / nu
logit_t = -np.log(1 + d2 / nu)

plt.figure(figsize=(8, 5))

# Plot the Logits
plt.plot(d2, logit_gaussian, label='Standard Logit (Linear Gain)', color='#E74C3C', linestyle='--')
plt.plot(d2, logit_t, label='AFA Logit (Logarithmic Compression)', color='#2980B9', linewidth=3)

# Highlight the Slope (Gain) near the origin
plt.arrow(0.5, -0.25, 0, 0.15, head_width=0.05, head_length=0.03, fc='k', ec='k')
plt.text(0.55, -0.2, 'AFA: Lower gain near noise floor', fontsize=10)

plt.xlabel(r'Inconsistency (Squared Mahalanobis Distance $d^2$)', fontsize=12)
plt.ylabel('Attention Energy (Logit)', fontsize=12)
plt.title('Logit Flatness: Handling the Noise Floor', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig('logit_flatness_origin.png', dpi=300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_mahalanobis_vs_euclidean():
    # 1. Setup Query and Keys
    query = np.array([0, 0])
    key_a = np.array([2, 0.5])   # "In-liers" according to dynamics
    key_b = np.array([0.5, 2])   # "Out-liers" according to dynamics
    
    # 2. Define Covariance from SDE (stretched along X-axis, e.g., high momentum)
    # This represents the V^C matrix derived from the DLE
    cov = np.array([[2.0, 0.5], 
                    [0.5, 0.2]]) 
    inv_cov = np.linalg.inv(cov)
    
    # 3. Calculate Distances
    # Euclidean
    d_euc_a = np.linalg.norm(key_a - query)
    d_euc_b = np.linalg.norm(key_b - query)
    
    # Mahalanobis: d^2 = r^T P r
    def mahalanobis_sq(k, q, P):
        r = k - q
        return r.T @ P @ r

    d_mah_a = mahalanobis_sq(key_a, query, inv_cov)
    d_mah_b = mahalanobis_sq(key_b, query, inv_cov)

    # 4. Plotting
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Plot Euclidean Circles (Standard Attention Bias)
    circle = plt.Circle((0, 0), d_euc_a, color='gray', fill=False, linestyle='--', label='Euclidean Equidistance')
    ax.add_patch(circle)
    
    # Plot Mahalanobis Ellipses (AFA Precision Prior)
    # We plot the 1, 2, and 3-sigma contours
    for n in [1, 2, 3]:
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:,order]
        theta = np.degrees(np.arctan2(*vecs[:,0][::-1]))
        width, height = 2 * n * np.sqrt(vals)
        ell = Ellipse(xy=query, width=width, height=height, angle=theta, 
                      edgecolor='#2980B9', fc='none', lw=2, alpha=0.6,
                      label=f'AFA Precision Prior ({n}$\sigma$)' if n==1 else "")
        ax.add_patch(ell)

    # Plot Points
    ax.scatter(*query, color='black', s=100, label='Query ($q_i$)', zorder=5)
    ax.scatter(*key_a, color='#27AE60', s=100, label=f'Key A ($d^2_{{mah}}={d_mah_a:.1f}$)', zorder=5)
    ax.scatter(*key_b, color='#E74C3C', s=100, label=f'Key B ($d^2_{{mah}}={d_mah_b:.1f}$)', zorder=5)

    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.set_title("Mahalanobis Distance: The Physical Consistency Metric", fontsize=14)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.2)
    
    plt.savefig('mahalanobis_importance.png', dpi=300)
    print(f"Key A Euclidean: {d_euc_a:.2f}, Key B Euclidean: {d_euc_b:.2f}")

plot_mahalanobis_vs_euclidean()

Figure: Mahalanobis Distance as a Physical Consistency Metric. While standard attention (gray circle) weights tokens based on isotropic Euclidean distance, AFA utilizes the Mahalanobis distance $d_{ij}^2 = \boldsymbol{r}_{ij}^\top \boldsymbol{P}_{ij}^C \boldsymbol{r}_{ij}$ to account for the structured uncertainty of the SDE. As illustrated, Key A and Key B are equidistantly located from the query in Euclidean space. However, AFA identifies Key A as statistically consistent with the propagated state covariance (blue ellipses), while Key B is identified as an outlier relative to the learned dynamics. This ensures that the attention mechanism is selective for tokens that follow a physically plausible trajectory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Define the state at current time i (The Query)
q = np.array([2.0, 0.0])

# Define a Key at past time j
# This key is physically distant but has a large magnitude
k_distal = np.array([2.5, 1.5]) 

# Define SDE transition (rotation and decay)
theta = np.radians(45)
decay = 0.6
Phi = decay * np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta),  np.cos(theta)]])

# Propagated Key (The "Pulled-Forward" Key)
k_hat = Phi @ k_distal

# Covariance at time i (DLE solution)
cov = np.array([[0.3, 0.05], [0.05, 0.1]])
inv_cov = np.linalg.inv(cov)

# Calculate Scores
dot_product = np.dot(q, k_distal)
residual = q - k_hat

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Panel 1: Standard Dot-Product Attention ---
ax1.quiver(0, 0, q[0], q[1], angles='xy', scale_units='xy', scale=1, color='black', label='Query $q_i$', width=0.015)
ax1.quiver(0, 0, k_distal[0], k_distal[1], angles='xy', scale_units='xy', scale=1, color='#E74C3C', label='Key $k_j$', width=0.015)

# Show projection line
proj_len = np.dot(k_distal, q) / np.linalg.norm(q)
proj_vec = (proj_len / np.linalg.norm(q)) * q
ax1.plot([k_distal[0], proj_vec[0]], [k_distal[1], proj_vec[1]], 'k--', alpha=0.5)
ax1.set_title("Standard Attention: Geometric Alignment", fontsize=14)
ax1.set_xlim(-0.5, 3.5); ax1.set_ylim(-1, 2.5)
ax1.set_aspect('equal'); ax1.grid(True, alpha=0.2); ax1.legend()

# --- Panel 2: AFA Mahalanobis Distance ---
ax2.quiver(0, 0, q[0], q[1], angles='xy', scale_units='xy', scale=1, color='black', label='Query $q_i$', width=0.015)
ax2.quiver(0, 0, k_distal[0], k_distal[1], angles='xy', scale_units='xy', scale=1, color='#E74C3C', alpha=0.2, label='Original $k_j$')
ax2.quiver(0, 0, k_hat[0], k_hat[1], angles='xy', scale_units='xy', scale=1, color='#2980B9', label='Pulled-Forward $\Phi k_j$', width=0.015)

# Show Residual
ax2.annotate('', xy=q, xytext=k_hat, arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax2.text(1.1, 0.4, 'Residual $r$', color='green', fontweight='bold')

# Plot Covariance Ellipse (The DLE Prior)
vals, vecs = np.linalg.eigh(cov)
order = vals.argsort()[::-1]
vals, vecs = vals[order], vecs[:,order]
theta_deg = np.degrees(np.arctan2(*vecs[:,0][::-1]))
width, height = 2 * 2 * np.sqrt(vals) 
ell = Ellipse(xy=q, width=width, height=height, angle=theta_deg, edgecolor='#2980B9', fc='none', lw=2, ls='--', label='SDE Uncertainty ($V^C$)')
ax2.add_patch(ell)

ax2.set_title("AFA: Dynamical Consistency (Mahalanobis)", fontsize=14)
ax2.set_xlim(-0.5, 3.5); ax2.set_ylim(-1, 2.5)
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.2); ax2.legend()

plt.tight_layout()
plt.savefig('alignment_vs_consistency.png', dpi=300)

Figure: Comparison of Attention Scoring Mechanisms. This visualization contrasts the standard dot-product heuristic with the AFA state-tracking approach for a distal token.Left (Standard Attention): Illustrates Geometric Alignment. A past key $\mathbf{k}_j$ (red) with high magnitude and partial directional alignment produces a large dot-product score via projection (dashed line). This mechanism ignores the underlying dynamics, making it susceptible to "attention hijacking" by high-norm tokens that may be temporally irrelevant.Right (AFA Attention): Illustrates Dynamical Consistency. The original key (faded red) is first propagated through the SDE transition matrix to its predicted position $\mathbf{\Phi}\mathbf{k}_j$ (blue). The attention weight is then determined by the residual $\mathbf{r}$ (green) relative to the query $\mathbf{q}_i$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

# 1. Setup Query and Dynamics
q = np.array([1.0, 1.0]) # Target state at time i
theta = np.radians(30)
decay = 0.8
Phi = decay * np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta),  np.cos(theta)]])

# 2. Define two competing Keys
# Key A: "The Correct Match" - After dynamics, it lands very close to q
k_a_orig = np.linalg.inv(Phi) @ (q + np.array([0.1, -0.1]))
k_a_prop = Phi @ k_a_orig

# Key B: "The Hijacker" - After dynamics, it is perfectly aligned with q 
# but has a massive magnitude (norm inflation)
k_b_prop = 3.0 * q # Overshoots significantly
k_b_orig = np.linalg.inv(Phi) @ k_b_prop

# 3. Compute Scores
# Method 1: Propagate + Dot Product
score_dp_a = np.dot(q, k_a_prop)
score_dp_b = np.dot(q, k_b_prop)
weights_dp = softmax([score_dp_a, score_dp_b])

# Method 2: Propagate + Residual Norm (Negative L2 Distance)
score_res_a = -np.linalg.norm(q - k_a_prop)**2
score_res_b = -np.linalg.norm(q - k_b_prop)**2
weights_res = softmax([score_res_a, score_res_b])

# 4. Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Vector Space (after dynamics)
ax1.quiver(0, 0, q[0], q[1], angles='xy', scale_units='xy', scale=1, color='black', label='Query $q_i$', width=0.02)
ax1.quiver(0, 0, k_a_prop[0], k_a_prop[1], angles='xy', scale_units='xy', scale=1, color='#2980B9', label='Propagated Key A (Match)', width=0.015)
ax1.quiver(0, 0, k_b_prop[0], k_b_prop[1], angles='xy', scale_units='xy', scale=1, color='#E74C3C', label='Propagated Key B (High Norm)', width=0.015)

# Circle representing "Ideal Match Zone"
circle = plt.Circle((q[0], q[1]), 0.5, color='green', fill=False, linestyle='--', alpha=0.5, label='Tracking Target')
ax1.add_patch(circle)

ax1.set_xlim(-0.5, 4.5)
ax1.set_ylim(-0.5, 4.5)
ax1.set_aspect('equal')
ax1.set_title("Propagated Keys in Latent Space", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.2)

# Panel 2: Comparison of Attention Weights
labels = ['Key A (Near)', 'Key B (Far/Aligned)']
x = np.arange(len(labels))
width = 0.35

ax2.bar(x - width/2, weights_dp, width, label='Propagated + Dot Product', color='#E74C3C', alpha=0.7)
ax2.bar(x + width/2, weights_res, width, label='Propagated + Residual Norm', color='#2980B9', alpha=0.7)

ax2.set_ylabel('Attention Weight', fontsize=12)
ax2.set_title('Attention Allocation: Hijacking vs. Tracking', fontsize=14)
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.legend()
ax2.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('residual_vs_dotproduct.png', dpi=300)
print(f"Weights (Dot Product): {weights_dp}")
print(f"Weights (Residual): {weights_res}")

Figure: Comparison of Scoring Metrics after SDE Dynamics. This figure demonstrates why applying SDE dynamics $\boldsymbol{\Phi}$ requires the Residual Norm rather than the standard Dot Product to maintain physical consistency.Left (Latent Space Visualization): After propagation through the SDE, Key A (blue) correctly tracks the Query's position, while Key B (red) is directionally aligned but suffers from "norm-inflation," overshooting the target state.Right (Attention Allocation): The bar chart reveals that the Propagated Dot Product (red) is hijacked by the magnitude of Key B, incorrectly assigning it $\approx 98\%$ of the attention weight. In contrast, the AFA Residual Norm (blue) correctly identifies Key A as the true dynamical match.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_afa_sandwich_diagram_stable():
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Styling definitions
    block_params = dict(edgecolor='#2C3E50', facecolor='#FBFCFC', lw=2)
    operator_params = dict(edgecolor='#E67E22', facecolor='#FEF5E7', lw=2) 
    core_params = dict(edgecolor='#2980B9', facecolor='#EBF5FB', lw=2.5) 
    arrow_params = dict(arrowstyle='->', lw=1.5, mutation_scale=20, color='#34495E')

    # 1. INPUT LAYER
    ax.text(0.5, 4.0, "Input Sequence\n" + r"$X \in \mathbb{R}^{N \times d}$", 
            ha='center', fontweight='bold', fontsize=12)

    # 2. ROTATION STEP (Top of Sandwich)
    ax.add_patch(patches.Rectangle((2.0, 3.0), 2.5, 2.0, **operator_params))
    ax.text(3.25, 4.0, "Eigenframe Rotation\n" + r"$(\tilde{\Phi}^- \odot Q, ...)$" + "\nGauge Mapping", 
            ha='center', va='center', fontweight='bold')
    
    # 3. THE N^2 CORE (The "Sandwich Filling")
    ax.add_patch(patches.FancyBboxPatch((5.5, 1.8), 4.5, 4.4, boxstyle="round,pad=0.3", **core_params))
    ax.text(7.75, 6.4, r"Isotropic AFA Core ($O(N^2 d)$)", ha='center', color='#2980B9', fontweight='bold', fontsize=13)
    
    # Sub-block: Residual Norm
    ax.add_patch(patches.Rectangle((6.0, 4.8), 3.5, 0.9, **block_params))
    ax.text(7.75, 5.25, r"$D^2 = f(Q, K, \mathrm{Re}(\tilde{Q}^\dagger \tilde{K}))$" + "\nResidual Norm", 
            ha='center', va='center', fontsize=10)
    
    # Sub-block: Robust M-Estimator
    ax.add_patch(patches.Rectangle((6.0, 3.4), 3.5, 0.9, **block_params))
    ax.text(7.75, 3.85, r"$L = \mathrm{Logits}(D^2, P_{\Delta t})$" + "\nM-Estimator Weight", 
            ha='center', va='center', fontsize=10)
    
    # Sub-block: Aggregation
    ax.add_patch(patches.Rectangle((6.0, 2.0), 3.5, 0.9, **block_params))
    ax.text(7.75, 2.45, r"$\hat{V} = (A \odot E) \tilde{V}^\top$" + "\nDecayed Aggregation", 
            ha='center', va='center', fontsize=10)
    
    # 4. REVERSE ROTATION (Bottom of Sandwich)
    ax.add_patch(patches.Rectangle((11.0, 3.0), 2.5, 2.0, **operator_params))
    ax.text(12.25, 4.0, "Inverse Rotation\n" + r"$\tilde{\Phi}^+ \odot \bar{V}$" + "\nPhase Restoration", 
            ha='center', va='center', fontweight='bold')
    
    # 5. OUTPUT
    ax.text(14.5, 4.0, "State Estimate\n" + r"$\bar{V} \in \mathbb{R}^{d \times N}$", 
            ha='center', fontweight='bold', fontsize=12)

    # Connective Arrows
    ax.annotate("", xy=(2.0, 4.0), xytext=(1.2, 4.0), arrowprops=arrow_params)
    ax.annotate("", xy=(5.5, 4.0), xytext=(4.5, 4.0), arrowprops=arrow_params)
    ax.annotate("", xy=(11.0, 4.0), xytext=(10.0, 4.0), arrowprops=arrow_params)
    ax.annotate("", xy=(14.2, 4.0), xytext=(13.5, 4.0), arrowprops=arrow_params)
    
    # Param Entry
    ax.annotate(r"Shared Decay $\mu$", xy=(7.75, 1.8), xytext=(7.75, 0.8), 
                ha='center', arrowprops=dict(arrowstyle='->', ls='--', color='gray'))

    ax.set_xlim(0, 16)
    ax.set_ylim(0, 7)
    ax.axis('off')
    plt.savefig('afa_sandwich_diagram_v2.png', dpi=300, bbox_inches='tight')
    plt.show()

draw_afa_sandwich_diagram_stable()

Figure A.4: Structural Flow of the Scalable Isotropic AFA. This diagram illustrates the hardware-efficient refactoring of the SDE observer into a "rotate-aggregate-rotate" sandwich.Top Slice (The Gauge Mapping): Input features are rotated by $\boldsymbol{\tilde{\Phi}}^{-}$ into a stationary eigenframe. This step is a pointwise $\mathcal{O}(Nd)$ operation that aligns the phase of all tokens, ensuring that the subsequent attention mechanism is equivariant to temporal shifts.The Core (Efficient Residuals): In the stationary frame, the $\mathcal{O}(N^2 d)$ bottleneck is isolated to a single matrix multiplication $\mathrm{Re}(\boldsymbol{\tilde{Q}}^{\dagger} \boldsymbol{\tilde{K}})$. Because the SDE noise and decay are assumed isotropic, the precision kernel $\boldsymbol{P}_{\Delta t}$ is a scalar, allowing the Mahalanobis distance $\boldsymbol{D}^2$ to be computed via broadcasting.Bottom Slice (The Phase Restoration): The aggregated estimates are rotated back using $\boldsymbol{\tilde{\Phi}}^{+}$. This restores the dynamical phase relationships, producing a state estimate $\bar{\boldsymbol{V}}$ that is physically consistent with the original SDE dynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def afa_components(t, mu, sigma, eta, gamma):
    # DLE Variance
    var = (sigma**2 * (1 - np.exp(-2 * mu * t)) / (2 * mu) + 
           eta**2 * np.exp(-2 * mu * t) + gamma**2)
    P = 1.0 / var
    B = np.log(P)
    return P, B

t = np.linspace(0, 5, 500)
# Setup Regimes
P_diff, B_diff = afa_components(t, mu=0.5, sigma=1.2, eta=0.2, gamma=0.1)
P_int, B_int = afa_components(t, mu=0.5, sigma=0.2, eta=1.5, gamma=0.1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot Multiplier P
ax1.plot(t, P_diff, color='tab:red', label='Diffusive $P$ (Selectivity)')
ax1.plot(t, P_int, color='tab:blue', label='Integrative $P$ (Selectivity)')
ax1.set_title("Multiplicative Multiplier ($P_{\Delta t}$)")
ax1.legend()

# Plot Bias B
ax2.plot(t, B_diff, '--', color='tab:red', label='Diffusive $B$ (Prior)')
ax2.plot(t, B_int, '--', color='tab:blue', label='Integrative $B$ (Prior)')
ax2.set_title("Additive Bias ($B_{\Delta t}$)")
ax2.legend()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Standardized publication settings
plt.rcParams.update({
    'font.size': 14, 
    'axes.titlesize': 18, 
    'axes.labelsize': 16,
    'legend.fontsize': 12,
    'lines.linewidth': 2.5
})

def get_afa_params(t, mu, sigma, eta, gamma):
    # V(t) calculation based on the paper's DLE solution
    var = (sigma**2 * (1 - np.exp(-2 * mu * t)) / (2 * mu) + 
           eta**2 * np.exp(-2 * mu * t) + gamma**2)
    P = 1.0 / var
    B = np.log(P)
    alpha = eta**2 - (sigma**2 / (2 * mu))
    beta = gamma**2 + (sigma**2 / (2 * mu))
    return P, B, alpha, beta

# Time vector
t = np.linspace(0, 10, 500)
mu = 0.5
gamma = 0.1

# Define Parameter Ranges
sigmas_diff = [0.2, 0.3, 0.5, 2.0]
etas_int = [0.5, 1.0, 2.0, 4.0]

# Define standardized color maps
reds = plt.cm.Reds(np.linspace(0.4, 1.0, len(sigmas_diff)))
blues = plt.cm.Blues(np.linspace(0.4, 1.0, len(etas_int)))

fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)

# --- Row 1: Diffusive Case Plotting (Varying Sigma) ---
eta_diff = 0.2
for i, s in enumerate(sigmas_diff):
    P, B, alpha, beta = get_afa_params(t, mu, s, eta_diff, gamma)
    label = f'$\\sigma$={s:.1f} ($\\alpha/\\beta$={alpha/beta:.2f})'
    axes[0, 0].plot(t, P, label=label, color=reds[i])
    axes[0, 1].plot(t, B, label=label, color=reds[i])

axes[0, 0].set_title("Diffusive ($P_{\\Delta t}$): Closing Gates")
axes[0, 1].set_title("Diffusive ($B_{\\Delta t}$): Forgetting Priors")
axes[0, 0].legend(loc='upper right', frameon=True)
axes[0, 1].legend(loc='upper right', frameon=True)

# --- Row 2: Integrative Case Plotting (Varying Eta) ---
sigma_int = 0.2
for i, e in enumerate(etas_int):
    P, B, alpha, beta = get_afa_params(t, mu, sigma_int, e, gamma)
    label = f'$\\eta$={e:.1f} ($\\alpha/\\beta$={alpha/beta:.2f})'
    axes[1, 0].plot(t, P, label=label, color=blues[i])
    axes[1, 1].plot(t, B, label=label, color=blues[i])

axes[1, 0].set_title("Integrative ($P_{\\Delta t}$): Opening Gates")
axes[1, 1].set_title("Integrative ($B_{\\Delta t}$): Settling Priors")
axes[1, 0].legend(loc='lower right', frameon=True)
axes[1, 1].legend(loc='lower right', frameon=True)

# Formatting
for i in range(2):
    axes[i, 0].set_ylabel("Precision $P$")
    axes[i, 1].set_ylabel("Log-Precision $B$")
    axes[1, i].set_xlabel("Time Lag $\\Delta t$")
    for j in range(2):
        axes[i, j].grid(True, linestyle=':', alpha=0.6)

plt.show()

By varying the ratio of process noise $\sigma$ to measurement noise $\eta$, AFA heads can specialize into distinct physical regimes: a diffusive regime that favors local recency and an integrative regime that filters transient noise to identify stable historical trends.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Publication-quality settings
plt.rcParams.update({
    'font.size': 14, 
    'axes.titlesize': 18, 
    'axes.labelsize': 16,
    'legend.fontsize': 12,
    'lines.linewidth': 2.5
})

def afa_kernel(t, mu, sigma, eta, gamma):
    if mu < 1e-5:
        var = sigma**2 * t + eta**2 + gamma**2
    else:
        var = (sigma**2 * (1 - np.exp(-2 * mu * t)) / (2 * mu) + 
               eta**2 * np.exp(-2 * mu * t) + gamma**2)
    P = 1.0 / var
    B = np.log(P)
    return P, B

t = np.linspace(0, 8, 500)
# mu_vals = [0.01, 0.2, 0.8, 2.0]
mu_vals = [2.0, 0.8, 0.2, 0.01]
gamma = 0.1

reds = plt.cm.Reds(np.linspace(0.4, 1.0, len(mu_vals)))
blues = plt.cm.Blues(np.linspace(0.4, 1.0, len(mu_vals)))

fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)

# --- Row 1: Diffusive Sweep (alpha < 0) ---
sigma_d, eta_d = 1.2, 0.2
for i, m in enumerate(mu_vals):
    P, B = afa_kernel(t, m, sigma_d, eta_d, gamma)
    axes[0, 0].plot(t, P, label=f'$\mu$={m}', color=reds[i])
    axes[0, 1].plot(t, B, label=f'$\mu$={m}', color=reds[i])

P_alibi, B_alibi = afa_kernel(t, 0.0, sigma_d, eta_d, gamma)
# axes[0, 1].plot(t, B_alibi, '--', color='black', alpha=0.6, label='Zero-decay limit ($\mu=0$)')
axes[0, 1].plot(t, B_alibi, '--', color='black', alpha=0.6, label='$\mu=0$')

# Legends for Top Row
axes[0, 0].legend(loc='upper right', frameon=True)
axes[0, 1].legend(loc='upper right', frameon=True)

# --- Row 2: Integrative Sweep (alpha > 0) ---
sigma_i, eta_i = 0.2, 1.5
for i, m in enumerate(mu_vals):
    P, B = afa_kernel(t, m, sigma_i, eta_i, gamma)
    axes[1, 0].plot(t, P, label=f'$\mu$={m}', color=blues[i])
    axes[1, 1].plot(t, B, label=f'$\mu$={m}', color=blues[i])

P_alibi_i, B_alibi_i = afa_kernel(t, 0.0, sigma_i, eta_i, gamma)
# axes[1, 1].plot(t, B_alibi_i, '--', color='black', alpha=0.6, label='Brownian Limit ($\mu=0$)')
axes[1, 1].plot(t, B_alibi_i, '--', color='black', alpha=0.6, label='$\mu=0$')

# Fix legend locations for Bottom Row to avoid covering the "settling" curves
axes[1, 0].legend(loc='lower right', frameon=True)
axes[1, 1].legend(loc='lower right', frameon=True)

# Titles and Labels
axes[0, 0].set_title("Diffusive Multiplier $P_{\Delta t}$ (Closing Gates)")
axes[0, 1].set_title("Diffusive Bias $B_{\Delta t}$ (Forgetting)")
axes[1, 0].set_title("Integrative Multiplier $P_{\Delta t}$ (Opening Gates)")
axes[1, 1].set_title("Integrative Bias $B_{\Delta t}$ (Settling)")

for i in range(2):
    axes[i, 0].set_ylabel("Precision $P$")
    axes[i, 1].set_ylabel("Log-Precision $B$")
    axes[1, i].set_xlabel("Time Lag $\Delta t$")

for ax in axes.flat:
    ax.grid(True, linestyle=':', alpha=0.6)

plt.show()

The damping parameter $\mu$ dictates the speed of the phase transition. At $\mu \to 0$, the model recovers non-stationary Brownian dynamics, where precision drops linearly with time. As $\mu$ increases, the model enforces stationarity, where the attention bias saturates to a learned global noise floor $\beta$, providing a principled mechanism for long-range context retention.